[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_07_active_media_waves.ipynb)


# 07. BZ chemistry, excitable media, and waves

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** How can chemical feedback generate oscillations locally and propagating excitation when diffusion is added?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- connect the chemistry of the Belousov–Zhabotinsky reaction to Oregonator variables and feedback loops
- simulate a stiff Oregonator model and distinguish model concentrations from observables
- distinguish bistable, excitable, and oscillatory local kinetics
- explain activator–inhibitor time-scale separation
- measure pulse speed and refractory effects in a spatial simulation


## Prerequisite bridge: nullclines, cycles, and excitability

For a two-variable system $\dot u=f(u,v)$, $\dot v=g(u,v)$, the curves $f=0$ and $g=0$ are nullclines; their intersections are fixed points. A stable limit cycle produces self-sustained oscillation without repeated external kicks. An excitable system instead has a stable resting point, but a perturbation beyond a threshold triggers a large excursion before recovery.

Fast activator and slow inhibitor nullclines explain both behaviors geometrically. In space, diffusion lets an excited region push its neighbor across threshold. The slow recovery variable then produces a refractory wake, selecting pulse direction and limiting how closely waves can follow.


## Belousov–Zhabotinsky reaction: chemistry before equations

The BZ reaction is a family of acid bromate oxidations of an organic substrate, commonly malonic acid, with a redox catalyst such as cerium or ferroin. It is a driven, dissipative chemical system while usable reactants remain. In a stirred batch reactor the oscillations ultimately die as reagents are consumed; a continuously fed stirred reactor can maintain the nonequilibrium state.

The Field–Kőrös–Noyes mechanism organizes the chemistry into coupled feedback loops. Bromous acid, $X=[\mathrm{HBrO_2}]$, participates in autocatalytic bromate chemistry and acts as a fast activator. Bromide, $Y=[\mathrm{Br^-}]$, suppresses that autocatalytic route and acts as an inhibitor. The oxidized catalyst, $Z$ (for example $[\mathrm{Ce^{IV}}]$), provides slower negative feedback through its coupling to the organic substrate. Alternation between bromide-rich inhibition and autocatalytic oxidation produces concentration and catalyst-color oscillations.

A standard dimensionless three-variable Oregonator reduction is

$$\varepsilon_x\dot x=qy-xy+x(1-x),$$
$$\varepsilon_y\dot y=-qy-xy+fz,$$
$$\dot z=x-z.$$

The variables are scaled concentrations, not direct color channels. The small $\varepsilon$ values encode separated chemical time scales. The Oregonator captures the feedback architecture and qualitative oscillation, but it is not a quantitatively complete mechanism for every BZ recipe.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

from scipy.integrate import solve_ivp

eps_x, eps_y, q, f = 9.9e-3, 1.98e-5, 7.62e-5, 1.0

def oregonator(t, state):
    x, y, z = state
    return ((q*y - x*y + x*(1-x))/eps_x,
            (-q*y - x*y + f*z)/eps_y,
            x-z)

# The chemistry is stiff: Radau is used rather than a large explicit step.
t_bz = np.linspace(0, 40, 5000)
bz = solve_ivp(oregonator, (0, 40), [0.1, 0.1, 0.1], t_eval=t_bz,
               method="Radau", rtol=2e-7, atol=1e-10)
if not bz.success:
    raise RuntimeError(bz.message)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(bz.t, bz.y[0], label=r"$x$: bromous-acid activator")
ax[0].plot(bz.t, bz.y[1], label=r"$y$: bromide inhibitor")
ax[0].plot(bz.t, bz.y[2], label=r"$z$: oxidized catalyst")
ax[0].set(xlabel="dimensionless time", ylabel="scaled concentration",
          title="Oregonator chemical oscillations")
ax[0].legend(fontsize=8)
ax[1].plot(bz.y[0], bz.y[2], lw=0.9)
ax[1].set(xlabel="activator x", ylabel="oxidized catalyst z",
          title="Attracting chemical cycle")
plt.tight_layout()


## From a well-mixed oscillator to chemical waves

In a well-stirred vessel, the state is approximately spatially uniform and the Oregonator trajectory describes temporal oscillation. In an unstirred thin layer, molecular diffusion couples neighboring volume elements. Local excitation can then recruit adjacent material, while slow recovery leaves a refractory wake. Target patterns and spirals are therefore spatiotemporal concentration fields, not material objects moving through the dish.

The FitzHugh–Nagumo equations below retain this activator–recovery geometry in a deliberately generic form. They are useful for wave dynamics but should not be presented as a detailed stoichiometric model of the BZ reaction.


## Activator–inhibitor dynamics

A dimensionless FitzHugh–Nagumo field is

$$\partial_tu=D_u\partial_x^2u+u-\frac{u^3}{3}-v+I,\qquad
\partial_tv=\varepsilon(u+a-bv).$$

The fast activator $u$ generates the leading front; the slow recovery variable $v$ raises the effective threshold behind it and creates refractoriness. Traveling pulses are homoclinic structures in the comoving coordinate $\xi=x-ct$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

N, L, dt, steps = 500, 120.0, 0.01, 12000
dx = L/N
x = np.linspace(0, L, N, endpoint=False)
Du, eps, a, b, I = 0.8, 0.08, 0.7, 0.8, 0.0
u = -1.2*np.ones(N)
v = -0.62*np.ones(N)
u[(x > 8) & (x < 16)] = 1.5

def lap_neumann(z):
    padded = np.pad(z, 1, mode="edge")
    return (padded[:-2] - 2*padded[1:-1] + padded[2:]) / dx**2

snapshots = []
for n in range(steps):
    u += dt*(Du*lap_neumann(u) + u - u**3/3 - v + I)
    v += dt*eps*(u + a - b*v)
    if n in [0, 2500, 5500, 9000, steps-1]:
        snapshots.append((n*dt, u.copy(), v.copy()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for t, us, vs in snapshots:
    ax[0].plot(x, us, label=f"t={t:.0f}")
ax[0].set(xlabel="x", ylabel="activator u", title="Traveling excitation")
ax[0].legend(ncol=2, fontsize=8)
ax[1].plot(snapshots[-1][1], snapshots[-1][2])
ax[1].set(xlabel="u", ylabel="v", title="Spatial profile in phase plane")
plt.tight_layout()


## Assessed exercise

Part A: estimate the Oregonator oscillation period from successive maxima of $z$, then vary $f$ and determine where sustained oscillations are lost; verify that the result is insensitive to solver tolerance. Part B: develop an automated FitzHugh–Nagumo pulse-position estimator and measure speed versus $D_u$, then estimate the refractory time using two stimuli. Explain which conclusions concern BZ chemistry and which concern only the generic excitable-medium model.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** Mikhailov, *Foundations of Synergetics I: Distributed Active Systems*, oscillatory and excitable media; Kuramoto, *Chemical Oscillations, Waves, and Turbulence*.

**Verified web reading:**
- [Scholarpedia: Belousov–Zhabotinsky reaction](https://www.scholarpedia.org/article/Belousov-Zhabotinsky_reaction)
- [Scholarpedia: Oregonator](https://www.scholarpedia.org/article/Oregonator)
- [Mikhailov's Foundations of Synergetics I](https://doi.org/10.1007/978-3-642-78556-6)
- [Scholarpedia: reaction-diffusion systems](https://www.scholarpedia.org/article/Reaction-diffusion_systems)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
